# Polar Format Algorithm (PFA) — SAR Image Formation from CPHD Phase History

**Objective**: Form a complex SAR image from Compensated Phase History Data (CPHD) using the Polar Format Algorithm (PFA).

## Overview

The **Polar Format Algorithm** is a frequency-domain SAR image formation technique that transforms phase history data sampled on a polar grid (range-Doppler space) into a rectangular image grid via interpolation and 2D inverse FFT.

## Theory

### Phase History to Image Formation

SAR phase history data $S(f_r, \theta)$ is collected in a polar coordinate system:
- $f_r$: range frequency (radial wavenumber)
- $\theta$: azimuth angle (platform track)

PFA performs these steps:

1. **Range Interpolation**: Resample radial samples onto a rectangular k-space grid (keystone formatting)
2. **Azimuth Interpolation**: Resample angular samples onto Cartesian grid
3. **2D IFFT**: Compress the rectangular k-space to image domain

### K-Space Grid

The polar grid is defined by:
$$k_r = \frac{4\pi f}{c}$$
where $f$ is transmit frequency and $c$ is speed of light.

The rectangular output grid requires **inscribed** (smaller footprint) or **circumscribed** (full coverage) fitting to the polar annulus.

### Resolution

Slant-plane resolution is determined by:
- **Range resolution**: $\rho_r = \frac{c}{2B}$ (bandwidth $B$)
- **Azimuth resolution**: $\rho_a = \frac{\lambda}{2\Delta\theta}$ (aperture angle $\Delta\theta$)

### Modules Used

| Module | Purpose |
|--------|--------|
| `grdl.IO` | CPHD reader via factory (`get_reader('cphd', ...)`) |
| `grdl.image_processing.sar` | `PolarFormatAlgorithm` |
| `napari` | Interactive image visualization |

## Preamble — Version Check

In [ ]:
import grdl
from packaging.version import parse as parse_version

required = "0.6.1"
current = grdl.__version__

if parse_version(current) < parse_version(required):
    raise RuntimeError(
        f"GRDL {required}+ required. Found {current}. "
        f"Update: pip install --upgrade grdl"
    )

print(f"✓ GRDL {current}")

## Imports

In [ ]:
from pathlib import Path

import numpy as np
import napari

from grdl.IO import get_reader
from grdl.image_processing.sar.image_formation import PolarFormatAlgorithm

## Configuration — User Paths

In [ ]:
# Path to CPHD file (spotlight collection)
CPHD_PATH = Path("/path/to/your/cphd_file.cphd")

# Validate path
assert CPHD_PATH.exists(), f"CPHD file not found: {CPHD_PATH}"
assert CPHD_PATH.suffix.lower() in [".cphd", ".cph"], f"Expected CPHD file, got {CPHD_PATH.suffix}"

print(f"✓ CPHD file: {CPHD_PATH.name}")

## Step 1: Load CPHD Metadata and Signal

In [ ]:
with get_reader('cphd', CPHD_PATH) as reader:
    meta = reader.metadata
    
    # Extract collection parameters
    channel = meta.data.channels[0]  # First channel
    n_vectors = channel.num_vectors
    n_samples = channel.num_samples
    
    print(f"Collection mode: {meta.data.radar_mode}")
    print(f"Phase history: {n_vectors} vectors × {n_samples} samples")
    print(f"Center frequency: {meta.data.tx_frequency_nominal / 1e9:.2f} GHz")
    print(f"Bandwidth: {meta.channel.parameters[0].bandwidth / 1e6:.1f} MHz")
    
    # Read signal vector
    signal = reader.read_signal(channel_id=channel.identifier)

print(f"\nSignal shape: {signal.shape}, dtype: {signal.dtype}")

## Step 2: Configure PFA Parameters

In [ ]:
# PFA algorithm configuration
pfa_params = {
    'grid_mode': 'inscribed',      # or 'circumscribed' for full coverage
    'range_oversample': 1.0,       # 1.0 = Nyquist sampling
    'azimuth_oversample': 1.0,
    'projection': 'slant',          # slant plane (default) or 'ground'
    'scene_sizing': 'full',         # 'full' = whole scene from data sampling
}

print("PFA Configuration:")
for k, v in pfa_params.items():
    print(f"  {k}: {v}")

## Step 3: Initialize PFA Processor

In [ ]:
with get_reader('cphd', CPHD_PATH) as reader:
    pfa = PolarFormatAlgorithm(
        metadata=reader.metadata,
        **pfa_params
    )
    
    print(f"PFA initialized")
    print(f"Output grid size: {pfa.grid.shape} (azimuth × range)")
    print(f"Pixel spacing: {pfa.grid.pixel_spacing_m} m (row, col)")

## Step 4: Run Image Formation Pipeline

PFA operates in three stages:
1. **Range interpolation**: Keystone formatting (polar radial → rectangular)
2. **Azimuth interpolation**: Angular → Cartesian resampling  
3. **Compression**: 2D IFFT to image domain

In [ ]:
with get_reader('cphd', CPHD_PATH) as reader:
    pfa = PolarFormatAlgorithm(metadata=reader.metadata, **pfa_params)
    signal = reader.read_signal()
    
    # Form the complex SAR image
    print("Running PFA image formation...")
    image = pfa.form(signal)

print(f"✓ Image formed: {image.shape}, dtype: {image.dtype}")

## Step 5: Compute Image Statistics

In [ ]:
# Magnitude statistics
magnitude = np.abs(image)
phase = np.angle(image)

print("Image Statistics:")
print(f"  Magnitude — min: {magnitude.min():.2e}, max: {magnitude.max():.2e}")
print(f"  Magnitude — mean: {magnitude.mean():.2e}, std: {magnitude.std():.2e}")
print(f"  Phase range: [{phase.min():.2f}, {phase.max():.2f}] rad")

# dB-scale magnitude
magnitude_db = 20 * np.log10(magnitude + 1e-12)
print(f"  dB — min: {magnitude_db.min():.1f}, max: {magnitude_db.max():.1f} dB")

## Step 6: Visualization — Interactive Napari Viewer

In [ ]:
viewer = napari.Viewer(title="PFA Image Formation")

# Layer 1: Magnitude (dB scale)
viewer.add_image(
    magnitude_db,
    name="Magnitude (dB)",
    colormap="gray",
    contrast_limits=[magnitude_db.max() - 50, magnitude_db.max()],
)

# Layer 2: Phase
viewer.add_image(
    phase,
    name="Phase (rad)",
    colormap="twilight",
    visible=False,
)

# Simplify viewer UI - hide unnecessary controls
viewer.window._qt_viewer.dockLayerControls.setVisible(False)  # Hide layer controls dock
viewer.window._qt_viewer.dockConsole.setVisible(False)  # Hide console
try:
    viewer.window._qt_viewer.activityDock.setVisible(False)  # Hide activity dock if present
except AttributeError:
    pass

print("✓ Napari viewer launched")
print("  Toggle layers: Magnitude (dB) / Phase (rad)")
print("  Adjust contrast: Right panel sliders")

## Physical Interpretation

### Magnitude Image
- **Bright pixels**: Strong backscatter (metal, corners, surfaces perpendicular to radar)
- **Dark pixels**: Low backscatter (smooth surfaces, vegetation, shadows)
- **dB scale**: Linear magnitude compressed to 50 dB dynamic range for display

### Phase Image
- Represents electromagnetic phase of backscattered signal
- Sensitive to sub-wavelength changes in range (interferometry, change detection)
- Uniformly distributed $[-\pi, \pi]$ for distributed scatterers

### Grid Modes
- **Inscribed**: Rectangular grid fits inside polar annulus → smaller scene, no invalid pixels
- **Circumscribed**: Rectangular grid encloses annulus → full coverage, corners may be invalid

### Resolution Trade-offs
- Larger bandwidth → better range resolution
- Larger aperture angle → better azimuth resolution
- Oversampling > 1.0 → finer pixels (zero-padding in k-space)

## Summary

Successfully demonstrated:
- ✅ CPHD phase history loading via IO factory pattern
- ✅ PFA algorithm initialization and configuration
- ✅ Three-stage image formation (range interp → azimuth interp → 2D IFFT)
- ✅ Complex SAR image with magnitude and phase
- ✅ Interactive napari visualization with dual layers

### Key GRDL Patterns
- `get_reader('cphd', path)` for format-agnostic CPHD loading
- Context manager (`with`) for automatic resource cleanup
- `PolarFormatAlgorithm.form(signal)` for end-to-end image formation
- Grid configuration via kwargs (grid_mode, oversampling, projection)

### Next Steps
- Try `grid_mode='circumscribed'` for full scene coverage
- Adjust oversampling factors (1.5, 2.0) for finer resolution
- Compare slant vs ground projection (`projection='ground'`)
- Export to SICD: `SICDWriter.write(image, metadata, output_path)`